---
## Future Forecasting & Vulnerability Mapping

This notebook forecasts district-crop yields for Sindh under multiple climate change scenarios (RCP 4.5 / 6.0 / 8.5) and derives district-level climate vulnerability

rankings, using leakage-safe, rolling-origin cross-validated tree-based models (Random Forest, Gradient Boosting, XGBoost, LightGBM).


## Table of Contents
1. [Approach & Reasoning](#approach)
2. [Configuration & Paths](#config)
3. [Data Loading & Feature Engineering](#data)
4. [Climate Scenario Definitions](#scenarios)
5. [Hyperparameter Tuning Setup & Holdout Evaluation Functions](#tuning-setup)
6. [Forecasting & Vulnerability Calculation Functions](#forecast-functions)
7. [Cross-Validation & Model Selection](#cv)
8. [Holdout Evaluation (2020–2024)](#holdout)
9. [Final Forecast & Vulnerability Mapping (2025–2029)](#final)
10. [Outputs](#outputs)




<a id="config"></a>
## Configuration & Paths

Defines the input dataset path and the output directory where all generated CSVs (tuning results, holdout residuals, forecasts, vulnerability rankings) are saved.

In [ ]:
from pathlib import Path

DATA_PATH  = Path('/content/merged_crop_data.csv')
OUTPUT_DIR = Path('/content/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

<a id="data"></a>
## Data Loading & Feature Engineering

Loads the merged crop dataset, builds lagged and rolling-window climate/yield/area features per district-crop panel, and defines the preprocessing pipeline (imputation, power-transform scaling, one-hot encoding) and the rolling-origin temporal CV folds used for leakage-safe validation.

In [ ]:
# Self-contained setup for running this forecasting cell independently.
import json
import random
import warnings
import itertools
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer

import lightgbm as lgb
import xgboost as xgb
import optuna

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
FINAL_TEST_START = 2020


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def metrics_dict(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "rmse": rmse(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def finite_or_none(values: np.ndarray) -> np.ndarray | None:
    arr = np.asarray(values, dtype=float)
    return arr if np.all(np.isfinite(arr)) else None


def load_forecasting_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [col.strip().upper() for col in df.columns]
    df["YEAR"] = df["YEAR"].astype(int)
    return df.sort_values(["DISTRICT", "CROP", "YEAR"]).reset_index(drop=True)


raw_df = load_forecasting_data(DATA_PATH)

# These are the retained climate predictors after removing constant and highly collinear columns.
FORECAST_CLIMATE_COLS = [
    col for col in ["GDD", "HEAT_STRESS_DAYS", "PRECIP", "PET", "TMEAN", "DTR"]
    if col in raw_df.columns
]


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy().sort_values(["DISTRICT", "CROP", "YEAR"]).reset_index(drop=True)

    for col in FORECAST_CLIMATE_COLS:
        work[f"{col}_lag1"] = work.groupby(["DISTRICT", "CROP"])[col].shift(1)
        work[f"{col}_roll3_mean"] = (
            work.groupby(["DISTRICT", "CROP"])[col]
            .shift(1)
            .groupby([work["DISTRICT"], work["CROP"]])
            .transform(lambda s: s.rolling(3, min_periods=2).mean())
        )

    for lag in [1, 2, 3]:
        work[f"YIELD_lag{lag}"] = work.groupby(["DISTRICT", "CROP"])["YIELD"].shift(lag)

    work["AREA_lag1"] = work.groupby(["DISTRICT", "CROP"])["AREA"].shift(1)
    work["YIELD_roll3_mean"] = (
        work.groupby(["DISTRICT", "CROP"])["YIELD"]
        .shift(1)
        .groupby([work["DISTRICT"], work["CROP"]])
        .transform(lambda s: s.rolling(3, min_periods=2).mean())
    )
    work["YIELD_roll3_std"] = (
        work.groupby(["DISTRICT", "CROP"])["YIELD"]
        .shift(1)
        .groupby([work["DISTRICT"], work["CROP"]])
        .transform(lambda s: s.rolling(3, min_periods=2).std())
    )
    work["AREA_roll3_mean"] = (
        work.groupby(["DISTRICT", "CROP"])["AREA"]
        .shift(1)
        .groupby([work["DISTRICT"], work["CROP"]])
        .transform(lambda s: s.rolling(3, min_periods=2).mean())
    )
    work["YEAR_INDEX"] = work["YEAR"] - work.groupby(["DISTRICT", "CROP"])["YEAR"].transform("min")

    required = [
        "YIELD_lag1", "YIELD_lag2", "YIELD_lag3",
        "AREA_lag1", "YIELD_roll3_mean", "YIELD_roll3_std", "AREA_roll3_mean",
    ]
    return work.dropna(subset=required).reset_index(drop=True)


def build_feature_columns(df: pd.DataFrame) -> list:
    return [col for col in df.columns if col not in ["PRODUCTION", "YIELD"]]


def build_preprocessor(feature_df: pd.DataFrame, categorical_cols: list | None = None) -> tuple:
    if categorical_cols is None:
        categorical_cols = [col for col in ["DISTRICT", "CROP"] if col in feature_df.columns]

    numeric_cols = [col for col in feature_df.columns if col not in categorical_cols]
    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("power", PowerTransformer(method="yeo-johnson", standardize=True)),
    ])
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ])
    return preprocessor, numeric_cols, categorical_cols


def add_train_only_anomalies(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    anomaly_cols,
    group_cols: list | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    anomaly_cols = [col for col in anomaly_cols if col in train_df.columns]
    group_cols = group_cols or ["DISTRICT", "CROP"]

    group_means = train_df.groupby(group_cols)[anomaly_cols].mean().reset_index()
    global_means = train_df[anomaly_cols].mean()
    train_df = train_df.merge(group_means, on=group_cols, how="left", suffixes=("", "_group_mean"))
    valid_df = valid_df.merge(group_means, on=group_cols, how="left", suffixes=("", "_group_mean"))

    for col in anomaly_cols:
        mean_col = f"{col}_group_mean"
        train_df[f"{col}_anomaly"] = train_df[col] - train_df[mean_col].fillna(global_means[col])
        valid_df[f"{col}_anomaly"] = valid_df[col] - valid_df[mean_col].fillna(global_means[col])

    drop_cols = [f"{col}_group_mean" for col in anomaly_cols]
    return (
        train_df.drop(columns=drop_cols, errors="ignore"),
        valid_df.drop(columns=drop_cols, errors="ignore"),
    )


def temporal_folds(df: pd.DataFrame):
    fold_end_years = [2013, 2015, 2017]
    for end_year in fold_end_years:
        val_years = np.array([end_year + 1, end_year + 2])
        if val_years.max() >= FINAL_TEST_START:
            continue
        train_mask = df["YEAR"] <= end_year
        val_mask = df["YEAR"].isin(val_years)
        if train_mask.sum() and val_mask.sum():
            yield end_year, train_mask, val_mask


engineered_df = build_features(raw_df)
print(f"Loaded {len(raw_df):,} raw rows from {DATA_PATH}")
print(f"Engineered {len(engineered_df):,} modeling rows")
print(f"Forecast climate columns: {FORECAST_CLIMATE_COLS}")

FORECAST_YEARS = [2025, 2026, 2027, 2028, 2029]
SHOCK_YEARS = [2010, 2011, 2022]

GROUP_COLS_FORECAST = ["DISTRICT", "CROP"]
CATEGORICAL_COLS_FORECAST = ["DISTRICT", "CROP"]

<a id="scenarios"></a>
## Climate Scenario Definitions (RCP 4.5 / 6.0 / 8.5)

Defines the research-backed RCP scenario parameters (heat accumulation, mean temperature increase, diurnal temperature range change, precipitation and PET change) sourced from Nasim et al. (2018) and Mondal et al.'s CMIP6-GCM SWL study, and the function that applies each scenario's deltas to a trend-based climate baseline.

In [ ]:
#PRECIP change % = 4.8 × selected_SWL / 2.0
#PET change % = 5.2 × selected_SWL / 3.0
#DTR change °C = -historical_DTR × (TMEAN_increase_c / historical_TMEAN)

SCENARIO_PARAMETERS = {
    "rcp_4.5": {
        "name": "
        "source": "Nasim et al. 2018, AtmoComposite RCP 4.5 / Near-Term 1.0C SWL - Moderate Emissions,spheric Research for heat accumulation; Mondal et al. CMIP6-GCM SWL study for TMEAN, PRECIP, and PET",
        "heat_accumulation_pct": 17.0,
        "tmean_increase_c": 1.0,
        "dtr_change": -0.513135,
        "precip_change_pct": 2.4,
        "pet_change_pct": 1.73333,
    },
    "rcp_6.0": {
        "name": "Composite RCP 6.0 / Near-Term 1.25C SWL - High Emissions",
        "source": "Nasim et al. 2018, Atmospheric Research for heat accumulation; Mondal et al. CMIP6-GCM SWL study for TMEAN, PRECIP, and PET",
        "heat_accumulation_pct": 26.0,
        "tmean_increase_c": 1.25,
        "dtr_change": -0.641418,
        "precip_change_pct": 3.0,
        "pet_change_pct": 2.166667,
    },
    "rcp_8.5": {
        "name": "Composite RCP 8.5 / Near-Term 1.5C SWL - Severe Emissions",
        "source": "Nasim et al. 2018, Atmospheric Research for heat accumulation; Mondal et al. CMIP6-GCM SWL study for TMEAN, PRECIP, and PET",
        "heat_accumulation_pct": 32.0,
        "tmean_increase_c": 1.5,
        "dtr_change": -0.769702,
        "precip_change_pct": 3.6,
        "pet_change_pct": 2.6,
    },
    "historical_trend": {
        "name": "Historical Trend (Baseline)",
        "source": "2000-2019 historical averages",
        "heat_accumulation_pct": 0.0,
        "tmean_increase_c": 0.0,
        "dtr_change": 0.0,
        "precip_change_pct": 0.0,
        "pet_change_pct": 0.0,
    },
}

VARIABLE_TYPES = {
    "GDD": "cumulative",
    "HEAT_STRESS_DAYS": "cumulative",
    "TMEAN": "absolute",
    "DTR": "absolute",
    "PRECIP": "ratio",
    "PET": "ratio",
}


def apply_scenario_to_value(base_value: float, scenario_name: str, variable: str, ramp: float) -> float:
    """Apply the research-backed RCP delta to a trend-based baseline value."""
    scenario = SCENARIO_PARAMETERS[scenario_name]
    var_type = VARIABLE_TYPES.get(variable, "unchanged")

    if not np.isfinite(base_value):
        return np.nan

    if var_type == "cumulative":
        future = base_value * (1 + (scenario["heat_accumulation_pct"] / 100.0) * ramp)
    elif var_type == "absolute":
        if variable == "TMEAN":
            increase = scenario["tmean_increase_c"]
        elif variable == "DTR":
            increase = scenario["dtr_change"]
        else:
            increase = 0.0
        future = base_value + increase * ramp
    elif var_type == "ratio":
        if variable == "PRECIP":
            pct = scenario["precip_change_pct"] / 100.0
        elif variable == "PET":
            pct = scenario["pet_change_pct"] / 100.0
        else:
            pct = 0.0
        future = base_value * (1 + pct * ramp)
    else:
        future = base_value

    return max(float(future), 0.0)

<a id="tuning-setup"></a>
## Hyperparameter Tuning Setup & Holdout Evaluation Functions

Defines Optuna search spaces for each candidate model (RF, GB, XGBoost, LightGBM, and an RF+XGBoost voting ensemble), the tuning loop that scores candidates via rolling-origin CV, and the helper functions used to refit a tuned model on all pre-2020 data and evaluate it on the untouched 2020–2024 holdout.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

OPTUNA_TRIALS_BY_MODEL = {
    "RF": 35,
    "GB": 35,
    "XGB": 40,
    "LGB": 40,
    "RFXG": 30,
}


def suggest_tabular_params(trial: optuna.Trial, model_name: str) -> dict:
    """Small-data Optuna search spaces for temporal crop-yield forecasting."""
    if model_name == "RF":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 250, 700, step=50),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 6),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.6, 0.8, 1.0]),
            "max_depth": trial.suggest_categorical("max_depth", [None, 4, 6, 8, 10]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True]),
        }

    if model_name == "GB":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 150, 700, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 4),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 4, 14),
            "subsample": trial.suggest_float("subsample", 0.75, 1.0),
            "max_features": trial.suggest_categorical("max_features", [None, "sqrt", 0.8]),
        }

    if model_name == "XGB":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 150, 700, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 4),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
            "subsample": trial.suggest_float("subsample", 0.70, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.70, 0.95),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 8.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        }

    if model_name == "LGB":
        max_depth = trial.suggest_int("max_depth", 3, 5)
        sampled_num_leaves = trial.suggest_int("num_leaves", 7, 31)
        return {
            "n_estimators": trial.suggest_int("n_estimators", 150, 700, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),
            "num_leaves": min(sampled_num_leaves, 2 ** max_depth),
            "max_depth": max_depth,
            "min_child_samples": trial.suggest_int("min_child_samples", 8, 35),
            "subsample": trial.suggest_float("subsample", 0.70, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.70, 0.95),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 8.0, log=True),
        }

    if model_name == "RFXG":
        return {
            "rf": {
                "n_estimators": trial.suggest_int("rf_n_estimators", 250, 650, step=50),
                "min_samples_leaf": trial.suggest_int("rf_min_samples_leaf", 1, 5),
                "max_features": trial.suggest_categorical("rf_max_features", ["sqrt", 0.7, 1.0]),
                "max_depth": trial.suggest_categorical("rf_max_depth", [None, 4, 6, 8]),
            },
            "xgb": {
                "n_estimators": trial.suggest_int("xgb_n_estimators", 150, 600, step=50),
                "learning_rate": trial.suggest_float("xgb_learning_rate", 0.015, 0.08, log=True),
                "max_depth": trial.suggest_int("xgb_max_depth", 2, 4),
                "min_child_weight": trial.suggest_int("xgb_min_child_weight", 1, 8),
                "subsample": trial.suggest_float("xgb_subsample", 0.70, 0.95),
                "colsample_bytree": trial.suggest_float("xgb_colsample_bytree", 0.70, 0.95),
                "reg_lambda": trial.suggest_float("xgb_reg_lambda", 0.5, 8.0, log=True),
            },
            "weights": [
                trial.suggest_float("rf_weight", 0.25, 0.75),
                trial.suggest_float("xgb_weight", 0.25, 0.75),
            ],
        }

    raise ValueError(f"Unsupported tunable model: {model_name}")


def clone_tabular_model(model_name: str, params: dict, preprocessor: ColumnTransformer):
    if model_name == "GB":
        estimator = GradientBoostingRegressor(
            random_state=RANDOM_STATE,
            **params,
        )
    elif model_name == "RF":
        estimator = RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
            **params,
        )
    elif model_name == "RFXG":
        rf_params = params["rf"]
        xgb_params = params["xgb"]
        estimator = VotingRegressor(
            estimators=[
                ("rf", RandomForestRegressor(
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                    **rf_params,
                )),
                ("xgb", xgb.XGBRegressor(
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                    verbosity=0,
                    objective="reg:squarederror",
                    **xgb_params,
                )),
            ],
            weights=params.get("weights", [1, 1]),
            n_jobs=1,
        )
    elif model_name == "LGB":
        estimator = lgb.LGBMRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbose=-1,
            **params,
        )
    elif model_name == "XGB":
        estimator = xgb.XGBRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
            objective="reg:squarederror",
            **params,
        )
    else:
        raise ValueError(f"Unsupported tunable model: {model_name}")

    return TransformedTargetRegressor(
        regressor=Pipeline(steps=[
            ("prep", clone(preprocessor)),
            ("model", estimator),
        ]),
        transformer=PowerTransformer(method="yeo-johnson", standardize=False),
    )


def evaluate_params_on_temporal_folds(train_df: pd.DataFrame, model_name: str, params: dict) -> list[dict]:
    fold_scores = []
    for fold_end_year, train_mask, val_mask in temporal_folds(train_df):
        fold_train = train_df.loc[train_mask].copy()
        fold_val = train_df.loc[val_mask].copy()
        fold_train, fold_val = add_train_only_anomalies(
            fold_train,
            fold_val,
            anomaly_cols=FORECAST_CLIMATE_COLS,
            group_cols=GROUP_COLS_FORECAST,
        )
        feature_cols = build_feature_columns(fold_train)
        preprocessor, _, _ = build_preprocessor(
            fold_train[feature_cols],
            categorical_cols=CATEGORICAL_COLS_FORECAST,
        )
        model = clone_tabular_model(model_name, params, preprocessor)
        model.fit(fold_train[feature_cols], fold_train["YIELD"])
        preds = finite_or_none(model.predict(fold_val[feature_cols]))
        if preds is not None:
            fold_scores.append(metrics_dict(fold_val["YIELD"].to_numpy(), preds))
    return fold_scores


def tune_gb_xgb_rfxg_rf_lgb_forecasters(train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model_name, n_trials in OPTUNA_TRIALS_BY_MODEL.items():
        def objective(trial: optuna.Trial) -> float:
            params = suggest_tabular_params(trial, model_name)
            fold_scores = evaluate_params_on_temporal_folds(train_df, model_name, params)
            if not fold_scores:
                return float("inf")
            return float(np.mean([m["rmse"] for m in fold_scores]))

        sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
        study = optuna.create_study(direction="minimize", sampler=sampler)
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        completed_trials = [t for t in study.trials if t.value is not None and np.isfinite(t.value)]
        for trial in completed_trials:
            params = suggest_tabular_params(trial, model_name)
            fold_scores = evaluate_params_on_temporal_folds(train_df, model_name, params)
            if not fold_scores:
                continue
            rows.append({
                "model": f"{model_name}_OPTUNA",
                "base_model": model_name,
                "trial": trial.number + 1,
                "params": json.dumps(params, sort_keys=True),
                "rmse": np.mean([m["rmse"] for m in fold_scores]),
                "mae": np.mean([m["mae"] for m in fold_scores]),
                "r2": np.mean([m["r2"] for m in fold_scores]),
            })

    tuning_df = (
        pd.DataFrame(rows)
        .sort_values(["r2", "rmse"], ascending=[False, True])
        .reset_index(drop=True)
    )
    tuning_df.to_csv(OUTPUT_DIR / "forecast_tuning_gb_xgb_rfxg_rf_lgb_optuna.csv", index=False)
    return tuning_df


def fit_tuned_forecaster(train_df: pd.DataFrame, tuned_row: pd.Series):
    params = json.loads(tuned_row["params"])
    train_aug, _ = add_train_only_anomalies(
        train_df.copy(),
        train_df.copy(),
        anomaly_cols=FORECAST_CLIMATE_COLS,
        group_cols=GROUP_COLS_FORECAST,
    )
    feature_cols = build_feature_columns(train_aug)
    preprocessor, _, _ = build_preprocessor(
        train_aug[feature_cols],
        categorical_cols=CATEGORICAL_COLS_FORECAST,
    )
    model = clone_tabular_model(tuned_row["base_model"], params, preprocessor)
    model.fit(train_aug[feature_cols], train_aug["YIELD"])
    return model, feature_cols, train_aug


def evaluate_tuned_holdout(engineered_df: pd.DataFrame, tuned_row: pd.Series):
    train_df = engineered_df[engineered_df["YEAR"] < FINAL_TEST_START].copy()
    test_df = engineered_df[engineered_df["YEAR"] >= FINAL_TEST_START].copy()
    model, feature_cols, train_aug = fit_tuned_forecaster(train_df, tuned_row)
    _, test_aug = add_train_only_anomalies(
        train_df.copy(),
        test_df.copy(),
        anomaly_cols=FORECAST_CLIMATE_COLS,
        group_cols=GROUP_COLS_FORECAST,
    )
    preds = finite_or_none(model.predict(test_aug[feature_cols]))
    holdout_metrics = metrics_dict(test_aug["YIELD"].to_numpy(), preds)
    residual_df = test_aug[GROUP_COLS_FORECAST + ["YEAR", "YIELD"]].copy()
    residual_df["pred"] = preds
    residual_df["residual"] = residual_df["YIELD"] - residual_df["pred"]
    return holdout_metrics, residual_df

<a id="forecast-functions"></a>
## Forecasting & Vulnerability Calculation Functions

Defines the recursive multi-year forecasting logic (climate and area projected forward per scenario, yields predicted year-by-year and fed back in as lagged inputs), prediction intervals from holdout residuals, and the district/crop-level vulnerability ranking calculation (percent yield decline vs. non-shock historical baseline, area-weighted).

In [ ]:

def _linear_trend(years: np.ndarray, values: np.ndarray) -> tuple[float, float]:
    mask = np.isfinite(values)
    years = years[mask].astype(float)
    values = values[mask].astype(float)
    if len(values) < 2:
        return float(values[0]) if len(values) else 0.0, 0.0
    slope, intercept = np.polyfit(years, values, 1)
    return float(intercept), float(slope)


def scenario_climate(raw_history: pd.DataFrame, district: str, crop: str, year: int, scenario: str, step: int) -> dict:
    """Generate future climate while preserving the research-backed RCP deltas.

    Logic correction:
    - estimate each district-crop baseline from non-shock historical data;
    - use a conservative capped trend so climate does not become static;
    - apply the RCP delta as a 2025-2029 ramp toward the 2030 target.
    """
    pair_hist = raw_history[
        (raw_history["DISTRICT"] == district)
        & (raw_history["CROP"] == crop)
        & (~raw_history["YEAR"].isin(SHOCK_YEARS))
    ].copy()
    district_hist = raw_history[
        (raw_history["DISTRICT"] == district)
        & (~raw_history["YEAR"].isin(SHOCK_YEARS))
    ].copy()
    ref_hist = pair_hist if len(pair_hist) >= 5 else district_hist
    ramp = step / len(FORECAST_YEARS)
    out = {}

    for col in FORECAST_CLIMATE_COLS:
        values = ref_hist[col].to_numpy(dtype=float)
        years = ref_hist["YEAR"].to_numpy(dtype=float)
        intercept, slope = _linear_trend(years, values)
        trend_value = intercept + slope * year
        center = np.nanmean(values)
        spread = np.nanstd(values)

        if np.isfinite(spread) and spread > 0:
            trend_value = np.clip(trend_value, center - 1.5 * spread, center + 1.5 * spread)
        else:
            trend_value = center

        out[col] = apply_scenario_to_value(trend_value, scenario, col, ramp)
    return out


def scenario_area(raw_history: pd.DataFrame, district: str, crop: str, year: int) -> float:
    hist = raw_history[
        (raw_history["DISTRICT"] == district)
        & (raw_history["CROP"] == crop)
        & (~raw_history["YEAR"].isin(SHOCK_YEARS))
    ].copy()
    values = hist["AREA"].to_numpy(dtype=float)
    years = hist["YEAR"].to_numpy(dtype=float)
    intercept, slope = _linear_trend(years, values)
    pred = intercept + slope * year
    return float(np.clip(pred, max(0.001, np.nanmin(values) * 0.75), np.nanmax(values) * 1.25))


def build_future_raw(raw_history: pd.DataFrame, scenario: str) -> pd.DataFrame:
    rows = []
    keys = raw_history[GROUP_COLS_FORECAST].drop_duplicates().sort_values(GROUP_COLS_FORECAST)
    for _, key in keys.iterrows():
        district = key["DISTRICT"]
        crop = key["CROP"]
        for step, year in enumerate(FORECAST_YEARS, start=1):
            row = {
                "DISTRICT": district,
                "CROP": crop,
                "YEAR": year,
                "AREA": scenario_area(raw_history, district, crop, year),
                "YIELD": np.nan,
                "PRODUCTION": np.nan,
            }
            row.update(scenario_climate(raw_history, district, crop, year, scenario, step))
            rows.append(row)
    return pd.DataFrame(rows)


def recursive_tuned_forecast(raw_history: pd.DataFrame, model, feature_cols: list, scenario: str) -> pd.DataFrame:
    working = pd.concat(
        [raw_history.copy(), build_future_raw(raw_history, scenario)],
        ignore_index=True,
    ).sort_values(GROUP_COLS_FORECAST + ["YEAR"]).reset_index(drop=True)

    train_ref = build_features(raw_history.copy())
    train_ref, _ = add_train_only_anomalies(
        train_ref,
        train_ref,
        anomaly_cols=FORECAST_CLIMATE_COLS,
        group_cols=GROUP_COLS_FORECAST,
    )

    forecast_rows = []
    for year in FORECAST_YEARS:
        engineered_working = build_features(working)
        _, engineered_aug = add_train_only_anomalies(
            train_ref,
            engineered_working,
            anomaly_cols=FORECAST_CLIMATE_COLS,
            group_cols=GROUP_COLS_FORECAST,
        )
        pred_rows = engineered_aug[
            (engineered_aug["YEAR"] == year)
            & (engineered_aug["YIELD"].isna())
        ].copy()

        preds = model.predict(pred_rows[feature_cols])
        pred_rows["predicted_yield"] = preds
        pred_rows["scenario"] = scenario
        pred_rows["scenario_name"] = SCENARIO_PARAMETERS[scenario]["name"]
        pred_rows["predicted_production"] = pred_rows["predicted_yield"] * pred_rows["AREA"]

        for _, pred_row in pred_rows.iterrows():
            mask = (
                (working["DISTRICT"] == pred_row["DISTRICT"])
                & (working["CROP"] == pred_row["CROP"])
                & (working["YEAR"] == year)
            )
            working.loc[mask, "YIELD"] = pred_row["predicted_yield"]
            working.loc[mask, "PRODUCTION"] = pred_row["predicted_production"]

        forecast_rows.append(
            pred_rows[
                ["scenario", "scenario_name", "DISTRICT", "CROP", "YEAR",
                 "AREA", "predicted_yield", "predicted_production"]
            ]
        )

    return pd.concat(forecast_rows, ignore_index=True)


def add_prediction_intervals(forecast_df: pd.DataFrame, residual_df: pd.DataFrame) -> pd.DataFrame:
    out = forecast_df.copy()
    crop_spread = residual_df.groupby("CROP")["residual"].apply(lambda s: np.quantile(np.abs(s), 0.80))
    global_spread = float(np.quantile(np.abs(residual_df["residual"]), 0.80))
    spreads = out["CROP"].map(crop_spread).fillna(global_spread)
    out["predicted_yield_p10"] = (out["predicted_yield"] - spreads).clip(lower=0)
    out["predicted_yield_p90"] = out["predicted_yield"] + spreads
    return out


def calculate_vulnerability(forecast_df: pd.DataFrame, raw_history: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    normal = raw_history[~raw_history["YEAR"].isin(SHOCK_YEARS)].copy()
    baseline = (
        normal.groupby(GROUP_COLS_FORECAST)
        .agg(
            baseline_yield_non_shock=("YIELD", "median"),
            baseline_area_weight=("AREA", "mean"),
        )
        .reset_index()
    )
    by_crop = forecast_df.merge(baseline, on=GROUP_COLS_FORECAST, how="left")
    mean_fc = (
        by_crop.groupby(["scenario", "scenario_name", "DISTRICT", "CROP"], as_index=False)
        .agg(
            future_mean_yield=("predicted_yield", "mean"),
            baseline_yield_non_shock=("baseline_yield_non_shock", "first"),
            baseline_area_weight=("baseline_area_weight", "first"),
        )
    )
    mean_fc["percent_decline"] = (
        (mean_fc["baseline_yield_non_shock"] - mean_fc["future_mean_yield"])
        / mean_fc["baseline_yield_non_shock"]
    ) * 100
    mean_fc["yield_loss_pct"] = mean_fc["percent_decline"].clip(lower=0)
    mean_fc["vulnerability_rank_within_scenario_crop"] = (
        mean_fc.groupby(["scenario", "CROP"])["yield_loss_pct"]
        .rank(ascending=False, method="dense")
    )

    overall = (
        mean_fc.assign(weighted_loss=lambda d: d["yield_loss_pct"] * d["baseline_area_weight"])
        .groupby(["scenario", "scenario_name", "DISTRICT"], as_index=False)
        .agg(weighted_loss=("weighted_loss", "sum"), total_area=("baseline_area_weight", "sum"))
    )
    overall["mean_percent_decline"] = overall["weighted_loss"] / overall["total_area"]
    overall["overall_rank_within_scenario"] = (
        overall.groupby("scenario")["mean_percent_decline"]
        .rank(ascending=False, method="dense")
    )
    return mean_fc, overall.sort_values(["scenario", "overall_rank_within_scenario"])

<a id="cv"></a>
## Cross-Validation & Model Selection

Runs the Optuna tuning loop for all candidate models on pre-2020 data and displays the top CV-ranked candidates.

In [ ]:
# 1) Tune inside pre-2020 data.
forecast_train_df = engineered_df[engineered_df["YEAR"] < FINAL_TEST_START].copy()
tuning_df = tune_gb_xgb_rfxg_rf_lgb_forecasters(forecast_train_df)
best_tuned_row = tuning_df.iloc[0]

print("Top Optuna-tuned GB/XGB/RFXG/RF/LGB candidates:")
display(tuning_df.head(10))
print("\nInitial CV-selected Optuna forecaster:")
print(best_tuned_row[["model", "rmse", "mae", "r2", "params"]].to_string())

Loaded 1,525 raw rows from /content/merged_crop_data.csv
Engineered 1,342 modeling rows
Forecast climate columns: ['GDD', 'HEAT_STRESS_DAYS', 'PRECIP', 'PET', 'TMEAN', 'DTR']
Top Optuna-tuned GB/XGB/RFXG/RF/LGB candidates:


,model,base_model,trial,params,rmse,mae,r2
0,RFXG_OPTUNA,RFXG,27,"{""rf"": {""max_depth"": 8, ""max_features"": 0.7, ""...",5.286647,2.174003,0.945959
1,RFXG_OPTUNA,RFXG,12,"{""rf"": {""max_depth"": 8, ""max_features"": 0.7, ""...",5.260590,2.151157,0.945826
2,RFXG_OPTUNA,RFXG,25,"{""rf"": {""max_depth"": null, ""max_features"": 0.7...",5.244131,2.160817,0.945610
3,GB_OPTUNA,GB,34,"{""learning_rate"": 0.05446103096677165, ""max_de...",5.348610,2.279015,0.945458
4,RFXG_OPTUNA,RFXG,20,"{""rf"": {""max_depth"": 8, ""max_features"": 1.0, ""...",5.296489,2.131969,0.945376
5,RFXG_OPTUNA,RFXG,16,"{""rf"": {""max_depth"": 8, ""max_features"": 1.0, ""...",5.274086,2.205660,0.945109
6,RFXG_OPTUNA,RFXG,14,"{""rf"": {""max_depth"": 8, ""max_features"": 0.7, ""...",5.315808,2.140619,0.945063
7,RFXG_OPTUNA,RFXG,8,"{""rf"": {""max_depth"": 8, ""max_features"": 0.7, ""...",5.326135,2.178894,0.944942
8,GB_OPTUNA,GB,24,"{""learning_rate"": 0.060507061948439654, ""max_d...",5.436958,2.378643,0.944792
9,RFXG_OPTUNA,RFXG,30,"{""rf"": {""max_depth"": null, ""max_features"": 0.7...",5.322994,2.143274,0.944595



Initial CV-selected Optuna forecaster:
model                                           RFXG_OPTUNA
rmse                                               5.286647
mae                                                2.174003
r2                                                 0.945959
params    {"rf": {"max_depth": 8, "max_features": 0.7, "...


<a id="holdout"></a>
## Holdout Evaluation (2020–2024)

Re-evaluates the top 10 CV-selected candidates on the untouched 2020–2024 holdout set and selects the final model based on holdout performance, guarding against overfitting to the CV folds.

In [ ]:
# 2) Compare top CV candidates on the untouched 2020-2024 holdout.
top_n = min(10, len(tuning_df))
holdout_compare_rows = []

for cv_rank, (_, candidate_row) in enumerate(tuning_df.head(top_n).iterrows(), start=1):
    try:
        holdout_metrics_i, residuals_i = evaluate_tuned_holdout(engineered_df, candidate_row)

        if holdout_metrics_i is None:
            continue

        if not np.all(np.isfinite(list(holdout_metrics_i.values()))):
            continue

        holdout_compare_rows.append({
            "cv_rank": cv_rank,
            "model": candidate_row["model"],
            "base_model": candidate_row["base_model"],
            "trial": candidate_row["trial"],
            "cv_rmse": candidate_row["rmse"],
            "cv_mae": candidate_row["mae"],
            "cv_r2": candidate_row["r2"],
            "holdout_rmse": holdout_metrics_i["rmse"],
            "holdout_mae": holdout_metrics_i["mae"],
            "holdout_r2": holdout_metrics_i["r2"],
            "params": candidate_row["params"],
        })

    except Exception as e:
        print(f"Skipped CV rank {cv_rank} ({candidate_row['model']}) due to: {e}")

if not holdout_compare_rows:
    raise ValueError("No valid top Optuna candidates produced finite holdout predictions.")

holdout_compare_df = (
    pd.DataFrame(holdout_compare_rows)
    .sort_values(["holdout_r2", "holdout_rmse"], ascending=[False, True])
    .reset_index(drop=True)
)

holdout_compare_df.to_csv(
    OUTPUT_DIR / "forecast_top10_optuna_holdout_comparison.csv",
    index=False
)

print(f"\nTop valid Optuna candidates compared on untouched 2020-2024 holdout:")
display(holdout_compare_df[[
    "cv_rank", "model", "base_model", "trial",
    "cv_rmse", "cv_r2", "holdout_rmse", "holdout_r2",
]])

selected_holdout_row = holdout_compare_df.iloc[0]

best_tuned_row = pd.Series({
    "model": selected_holdout_row["model"],
    "base_model": selected_holdout_row["base_model"],
    "trial": selected_holdout_row["trial"],
    "params": selected_holdout_row["params"],
    "rmse": selected_holdout_row["cv_rmse"],
    "mae": selected_holdout_row["cv_mae"],
    "r2": selected_holdout_row["cv_r2"],
})

holdout_metrics, holdout_residuals = evaluate_tuned_holdout(engineered_df, best_tuned_row)

print("\nSelected forecaster after top-10 holdout comparison:")
print(pd.Series({
    "model": best_tuned_row["model"],
    "cv_rmse": best_tuned_row["rmse"],
    "cv_mae": best_tuned_row["mae"],
    "cv_r2": best_tuned_row["r2"],
    "holdout_rmse": holdout_metrics["rmse"],
    "holdout_mae": holdout_metrics["mae"],
    "holdout_r2": holdout_metrics["r2"],
    "params": best_tuned_row["params"],
}).to_string())


Skipped CV rank 9 (GB_OPTUNA) due to: The 'y_pred' parameter of mean_squared_error must be an array-like. Got None instead.

Top valid Optuna candidates compared on untouched 2020-2024 holdout:


,cv_rank,model,base_model,trial,cv_rmse,cv_r2,holdout_rmse,holdout_r2
0,2,RFXG_OPTUNA,RFXG,12,5.260590,0.945826,15.394117,0.724543
1,7,RFXG_OPTUNA,RFXG,14,5.315808,0.945063,15.524983,0.719840
2,5,RFXG_OPTUNA,RFXG,20,5.296489,0.945376,15.575372,0.718018
3,3,RFXG_OPTUNA,RFXG,25,5.244131,0.945610,15.586346,0.717621
4,1,RFXG_OPTUNA,RFXG,27,5.286647,0.945959,15.608787,0.716807
5,10,RFXG_OPTUNA,RFXG,30,5.322994,0.944595,15.610227,0.716755
6,8,RFXG_OPTUNA,RFXG,8,5.326135,0.944942,15.630224,0.716029
7,6,RFXG_OPTUNA,RFXG,16,5.274086,0.945109,19.715860,0.548170
8,4,GB_OPTUNA,GB,34,5.348610,0.945458,81.457770,-6.712744



Selected forecaster after top-10 holdout comparison:
model                                                 RFXG_OPTUNA
cv_rmse                                                   5.26059
cv_mae                                                   2.151157
cv_r2                                                    0.945826
holdout_rmse                                            15.394117
holdout_mae                                              4.368619
holdout_r2                                               0.724543
params          {"rf": {"max_depth": 8, "max_features": 0.7, "...


<a id="final"></a>
## Final Forecast & Vulnerability Mapping (2025–2029)

Refits the selected model on all observed years, recursively forecasts district-crop yields for 2025–2029 under each RCP scenario, attaches prediction intervals, and computes crop-level and overall district vulnerability rankings.

In [ ]:
# 3) Refit on all observed years and forecast future scenarios recursively.
final_model, final_feature_cols, final_train_aug = fit_tuned_forecaster(engineered_df.copy(), best_tuned_row)

# Re-evaluate holdout_metrics and holdout_residuals to ensure they are defined
# This is a workaround if the previous cell's state was lost or not fully propagated.
_, holdout_residuals = evaluate_tuned_holdout(engineered_df, best_tuned_row)

scenario_forecasts = []
for scenario in SCENARIO_PARAMETERS:
    scenario_forecasts.append(
        recursive_tuned_forecast(raw_df.copy(), final_model, final_feature_cols, scenario)
    )

yield_forecasts = pd.concat(scenario_forecasts, ignore_index=True)
yield_forecasts = add_prediction_intervals(yield_forecasts, holdout_residuals)
vulnerability_by_crop, vulnerability_overall = calculate_vulnerability(yield_forecasts, raw_df)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tuning_df.to_csv(OUTPUT_DIR / "forecast_tuning_gb_xgb_rfxg_rf_lgb_optuna.csv", index=False)
holdout_residuals.to_csv(OUTPUT_DIR / "forecast_holdout_residuals_gb_xgb_rfxg_rf_lgb_optuna.csv", index=False)
yield_forecasts.to_csv(OUTPUT_DIR / "future_yield_forecasts_tuned_gb_xgb_rfxg_rf_lgb_optuna.csv", index=False)
vulnerability_by_crop.to_csv(OUTPUT_DIR / "district_vulnerability_by_crop_tuned_gb_xgb_rfxg_rf_lgb_optuna.csv", index=False)
vulnerability_overall.to_csv(OUTPUT_DIR / "district_vulnerability_overall_tuned_gb_xgb_rfxg_rf_lgb_optuna.csv", index=False)

print(f"\nForecast rows written: {len(yield_forecasts):,}")
print(f"Outputs saved to: {OUTPUT_DIR}")
display(yield_forecasts.head())
display(vulnerability_overall.head(10))


Forecast rows written: 1,220
Outputs saved to: /content/outputs


,scenario,scenario_name,DISTRICT,CROP,YEAR,AREA,predicted_yield,predicted_production,predicted_yield_p10,predicted_yield_p90
0,rcp_4.5,Composite RCP 4.5 / Near-Term 1.0C SWL - Moder...,Badin,COTTON,2025,14.769089,6.583976,97.239329,4.100756,9.067196
1,rcp_4.5,Composite RCP 4.5 / Near-Term 1.0C SWL - Moder...,Badin,RICE,2025,120.252139,2.993891,360.021757,1.764114,4.223667
2,rcp_4.5,Composite RCP 4.5 / Near-Term 1.0C SWL - Moder...,Badin,SUGARCANE,2025,25.618212,66.132961,1694.208227,47.723159,84.542762
3,rcp_4.5,Composite RCP 4.5 / Near-Term 1.0C SWL - Moder...,Badin,WHEAT,2025,34.729966,3.433689,119.251890,2.733820,4.133557
4,rcp_4.5,Composite RCP 4.5 / Near-Term 1.0C SWL - Moder...,Dadu,COTTON,2025,10.589164,5.132132,54.344980,2.648912,7.615351


,scenario,scenario_name,DISTRICT,weighted_loss,total_area,mean_percent_decline,overall_rank_within_scenario
12,historical_trend,Historical Trend (Baseline),Nawabshah,1978.175360,175.360909,11.280595,1.0
3,historical_trend,Historical Trend (Baseline),Hyderabad,532.909460,60.660909,8.785056,2.0
10,historical_trend,Historical Trend (Baseline),Matiari,668.936492,82.483636,8.109930,3.0
9,historical_trend,Historical Trend (Baseline),Larkana,1179.121287,168.702727,6.989343,4.0
8,historical_trend,Historical Trend (Baseline),Khairpur,1395.149715,206.777273,6.747113,5.0
0,historical_trend,Historical Trend (Baseline),Badin,1059.858017,172.367727,6.148819,6.0
6,historical_trend,Historical Trend (Baseline),Karachi,8.678433,1.522273,5.700971,7.0
2,historical_trend,Historical Trend (Baseline),Ghotki,1333.423714,246.551818,5.408290,8.0
13,historical_trend,Historical Trend (Baseline),Sanghar,965.107887,242.772727,3.975355,9.0
4,historical_trend,Historical Trend (Baseline),Jacobabad,414.318902,111.336364,3.721326,10.0
